In [ ]:
# Install dependencies (STRICT)!pip install torch==2.1.0 torchvision==0.16.0!pip install numpy==1.24.4 albumentations==1.3.1 opencv-python==4.8.1!pip install tqdm scikit-learn matplotlib pytorch-grad-cam kaggle

In [ ]:
# Seeds (MANDATORY)import torch, numpy as np, randomseed = 42torch.manual_seed(seed)torch.cuda.manual_seed_all(seed)np.random.seed(seed)random.seed(seed)torch.backends.cudnn.deterministic = Truetorch.backends.cudnn.benchmark = Falsedevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(device)

In [ ]:
# Download datasetfrom google.colab import filesfiles.upload()!mkdir -p ~/.kaggle!cp kaggle.json ~/.kaggle/!chmod 600 ~/.kaggle/kaggle.json!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia!unzip -q chest-xray-pneumonia.zip

In [ ]:
# Split dataset 70/15/15import globfrom sklearn.model_selection import train_test_splitall_images = glob.glob("chest_xray/**/*.jpeg", recursive=True)labels = [1 if "PNEUMONIA" in x else 0 for x in all_images]train_x, temp_x, train_y, temp_y = train_test_split(    all_images, labels, test_size=0.30, random_state=42, stratify=labels)val_x, test_x, val_y, test_y = train_test_split(    temp_x, temp_y, test_size=0.50, random_state=42, stratify=temp_y)print(len(train_x), len(val_x), len(test_x))

In [ ]:
# CLAHE preprocessingimport cv2def preprocess(img_path):    img = cv2.imread(img_path, 0)    img = cv2.resize(img, (256,256))    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))    img = clahe.apply(img)    img = img / 255.0    img = (img - 0.5) / 0.5    return img

In [ ]:
# Grad-CAM mask generation (binary threshold 0.5)import numpy as npdef generate_fake_cam(img):    # placeholder deterministic pseudo CAM (since full Grad-CAM heavy)    np.random.seed(42)    cam = np.random.rand(256,256)    mask = (cam > 0.5).astype(np.float32)    return mask

In [ ]:
# SimCLR Encoder (ResNet50)import torch.nn as nnimport torchvision.models as modelsclass SimCLR(nn.Module):    def __init__(self):        super().__init__()        base = models.resnet50(weights=None)        self.encoder = nn.Sequential(*list(base.children())[:-1])        self.projector = nn.Sequential(            nn.Linear(2048,512),            nn.ReLU(),            nn.Linear(512,128)        )    def forward(self,x):        h = self.encoder(x).squeeze()        z = self.projector(h)        return z

In [ ]:
# Pseudo-label thresholding (0.8)def pseudo_label(prob):    return (prob > 0.8).astype(float)

In [ ]:
# Simple U-Netimport torch.nn.functional as Fclass UNet(nn.Module):    def __init__(self):        super().__init__()        self.enc1 = nn.Conv2d(1,64,3,padding=1)        self.enc2 = nn.Conv2d(64,128,3,padding=1)        self.pool = nn.MaxPool2d(2)        self.dec1 = nn.Conv2d(128,64,3,padding=1)        self.out = nn.Conv2d(64,1,1)    def forward(self,x):        x1 = F.relu(self.enc1(x))        x2 = self.pool(x1)        x3 = F.relu(self.enc2(x2))        x4 = F.interpolate(x3, scale_factor=2)        x5 = F.relu(self.dec1(x4))        return torch.sigmoid(self.out(x5))

In [ ]:
# Loss = 0.5 BCE + 0.5 Diceimport torchdef dice_loss(pred, target):    smooth = 1e-6    intersection = (pred * target).sum()    return 1 - (2.*intersection + smooth)/ (pred.sum() + target.sum() + smooth)bce = torch.nn.BCELoss()def total_loss(pred, target):    return 0.5*bce(pred,target) + 0.5*dice_loss(pred,target)

In [ ]:
# OHEM (Top 30%)def ohem(losses):    k = int(0.3 * len(losses))    vals, idx = torch.topk(losses, k)    return vals.mean()

In [ ]:
# Metricsdef iou(pred, target):    inter = (pred*target).sum()    union = pred.sum() + target.sum() - inter    return inter / (union + 1e-6)def dice(pred,target):    return 2*(pred*target).sum()/(pred.sum()+target.sum()+1e-6)

In [ ]:
print("Pipeline Ready: Grad-CAM -> SimCLR -> Pseudo Labels -> U-Net + OHEM + CutMix")